# Rename Training WAV Files

This notebook previews and renames training `.wav` files to:

`[xx]-[type]-[yy]-[rec]-[zzzz].wav`

- `xx`: dB level (`-6`, `+0`, `+6`)
- `type`: machine type (e.g. `pump`, `valve`)
- `yy`: numeric part of `id_nn`
- `rec`: `abn` or `nml` from `abnormal/normal`
- `zzzz`: last 4 chars of original filename stem

Renaming executes only when `PROCESS = True`.


In [7]:
from pathlib import Path
import re
from uuid import uuid4

import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, display
from ipyfilechooser import FileChooser

PROCESS = True

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'datasets').exists() and (candidate / 'preprocessing').exists():
            return candidate
    raise RuntimeError('Could not infer repo root. Please open notebook from inside the repo tree.')


def _chooser_start_dir(path: Path) -> Path:
    if path.exists():
        return path
    for candidate in [path.parent, Path.cwd().resolve()]:
        if candidate.exists():
            return candidate
    return Path.cwd().resolve()


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEFAULT_DATA_ROOT = REPO_ROOT / 'datasets'
DISPLAY_LIMIT = 200

input_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_DATA_ROOT)))
input_dir_chooser.title = '<b>Input directory (db/type -> id_nn -> normal|abnormal)</b>'
input_dir_chooser.show_only_dirs = True
input_dir_chooser.use_dir_icons = True
input_dir_chooser.select_default = True

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))
display(input_dir_chooser)
print(f'PROCESS: {PROCESS}')
print(f'DISPLAY_LIMIT: {DISPLAY_LIMIT}')


Using repo root: `/home/mitch/development/raccoon-ball`

FileChooser(path='/home/mitch/development/raccoon-ball/datasets', filename='', title='<b>Input directory (db/t…

PROCESS: True
DISPLAY_LIMIT: 200


In [8]:
def resolve_selected_dir(chooser: FileChooser, fallback: Path) -> Path:
    selected = getattr(chooser, 'selected', None)
    if selected:
        return Path(selected).expanduser().resolve()
    return fallback.expanduser().resolve()


def _parse_db_and_type(db_type_dir: str) -> tuple[str, str]:
    m = re.fullmatch(r'([+-]?\d+)db-(.+)', db_type_dir)
    if not m:
        raise ValueError(f'Expected <db>db-<type>, got: {db_type_dir}')
    db_raw, machine_type = m.groups()
    xx = f'{int(db_raw):+d}'
    return xx, machine_type


def _parse_id(id_dir: str) -> str:
    m = re.fullmatch(r'id_(\d+)', id_dir)
    if not m:
        raise ValueError(f'Expected id_<digits>, got: {id_dir}')
    return m.group(1)


def _parse_rec(rec_dir: str) -> str:
    rec_map = {'normal': 'nm', 'abnormal': 'ab'}
    key = rec_dir.lower()
    if key not in rec_map:
        raise ValueError(f'Expected normal/abnormal directory, got: {rec_dir}')
    return rec_map[key]


def _extract_triplet_from_full_path(wav_path: Path) -> tuple[str, str, str]:
    # Find <db>db-<type> / id_<nn> / (normal|abnormal) anywhere in the absolute path
    # so DATA_ROOT can point to datasets, db/type, id_nn, or normal/abnormal subfolders.
    parts = wav_path.resolve().parts
    for i in range(len(parts) - 2):
        db_type_dir = parts[i]
        id_dir = parts[i + 1]
        rec_dir = parts[i + 2]
        if re.fullmatch(r'([+-]?\d+)db-(.+)', db_type_dir) and re.fullmatch(r'id_(\d+)', id_dir) and rec_dir.lower() in {'normal', 'abnormal'}:
            return db_type_dir, id_dir, rec_dir
    raise ValueError(
        'Could not locate <db>db-<type>/id_<nn>/(normal|abnormal) in path: '
        f'{wav_path}'
    )


def _build_new_name(wav_path: Path) -> tuple[str, dict]:
    db_type_dir, id_dir, rec_dir = _extract_triplet_from_full_path(wav_path)

    xx, machine_type = _parse_db_and_type(db_type_dir)
    yy = _parse_id(id_dir)
    rec = _parse_rec(rec_dir)

    stem = wav_path.stem
    zzzz = stem[-4:] if len(stem) >= 4 else stem.zfill(4)
    new_name = f'{xx}-{machine_type}-{yy}-{rec}-{zzzz}.wav'

    parts = {'xx': xx, 'type': machine_type, 'yy': yy, 'rec': rec, 'zzzz': zzzz}
    return new_name, parts


DATA_ROOT = resolve_selected_dir(input_dir_chooser, DEFAULT_DATA_ROOT)
print(f'DATA_ROOT: {DATA_ROOT}')

wav_files = sorted(p for p in DATA_ROOT.rglob('*.wav') if p.is_file())

base_columns = [
    'original_path', 'original_name', 'relative_path',
    'new_name', 'new_path', 'new_relative_path',
    'xx', 'type', 'yy', 'rec', 'zzzz',
    'action', 'error',
]
rows = []

for wav_path in wav_files:
    rel = wav_path.resolve().relative_to(DATA_ROOT.resolve())
    row = {
        'original_path': str(wav_path),
        'original_name': wav_path.name,
        'relative_path': rel.as_posix(),
        'new_name': None,
        'new_path': None,
        'new_relative_path': None,
        'xx': None,
        'type': None,
        'yy': None,
        'rec': None,
        'zzzz': None,
        'action': 'error',
        'error': None,
    }

    try:
        new_name, parts = _build_new_name(wav_path)
        new_path = wav_path.with_name(new_name)
        new_rel = new_path.resolve().relative_to(DATA_ROOT.resolve())

        row.update(parts)
        row['new_name'] = new_name
        row['new_path'] = str(new_path)
        row['new_relative_path'] = new_rel.as_posix()
        row['action'] = 'unchanged' if wav_path.name == new_name else 'rename'
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'

    rows.append(row)

PREVIEW_DF = pd.DataFrame(rows, columns=base_columns)

rename_count = int((PREVIEW_DF['action'] == 'rename').sum()) if not PREVIEW_DF.empty else 0
unchanged_count = int((PREVIEW_DF['action'] == 'unchanged').sum()) if not PREVIEW_DF.empty else 0
error_count = int(PREVIEW_DF['error'].notna().sum()) if not PREVIEW_DF.empty else 0

print(f'Total WAV files discovered: {len(PREVIEW_DF)}')
print(f'Rename candidates: {rename_count}')
print(f'Unchanged: {unchanged_count}')
print(f'Errors: {error_count}')

display_cols = ['relative_path', 'original_name', 'new_name', 'new_relative_path', 'action', 'error']
display(PREVIEW_DF[display_cols].head(DISPLAY_LIMIT))


DATA_ROOT: /home/mitch/development/raccoon-ball/datasets/fan
Total WAV files discovered: 16650
Rename candidates: 5550
Unchanged: 0
Errors: 11100


,relative_path,original_name,new_name,new_relative_path,action,error
0,+0db-fan/id_00/abnormal/00000000.wav,00000000.wav,+0-fan-00-ab-0000.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0000.wav,rename,NaN
1,+0db-fan/id_00/abnormal/00000001.wav,00000001.wav,+0-fan-00-ab-0001.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0001.wav,rename,NaN
2,+0db-fan/id_00/abnormal/00000002.wav,00000002.wav,+0-fan-00-ab-0002.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0002.wav,rename,NaN
3,+0db-fan/id_00/abnormal/00000003.wav,00000003.wav,+0-fan-00-ab-0003.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0003.wav,rename,NaN
4,+0db-fan/id_00/abnormal/00000004.wav,00000004.wav,+0-fan-00-ab-0004.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0004.wav,rename,NaN
...,...,...,...,...,...,...
195,+0db-fan/id_00/abnormal/00000195.wav,00000195.wav,+0-fan-00-ab-0195.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0195.wav,rename,NaN
196,+0db-fan/id_00/abnormal/00000196.wav,00000196.wav,+0-fan-00-ab-0196.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0196.wav,rename,NaN
197,+0db-fan/id_00/abnormal/00000197.wav,00000197.wav,+0-fan-00-ab-0197.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0197.wav,rename,NaN
198,+0db-fan/id_00/abnormal/00000198.wav,00000198.wav,+0-fan-00-ab-0198.wav,+0db-fan/id_00/abnormal/+0-fan-00-ab-0198.wav,rename,NaN


In [9]:
if 'PREVIEW_DF' not in globals():
    raise RuntimeError('Run the preview cell first to build PREVIEW_DF.')

if not PROCESS:
    print('PROCESS is False. No files were renamed.')
else:
    errors_df = PREVIEW_DF[PREVIEW_DF['error'].notna()]
    if not errors_df.empty:
        raise RuntimeError(f'Cannot rename while preview has errors. error_rows={len(errors_df)}')

    candidates = PREVIEW_DF[PREVIEW_DF['action'] == 'rename'].copy()
    if candidates.empty:
        print('No rename candidates found. Nothing to do.')
    else:
        source_paths = [Path(p) for p in candidates['original_path']]
        dest_paths = [Path(p) for p in candidates['new_path']]

        dest_series = pd.Series([str(p) for p in dest_paths])
        duplicate_dest = dest_series[dest_series.duplicated(keep=False)]
        if not duplicate_dest.empty:
            raise RuntimeError(
                'Duplicate destination names detected. Aborting. Example duplicates: '
                + ', '.join(sorted(duplicate_dest.unique())[:10])
            )

        source_set = {p.resolve() for p in source_paths}
        blocking = []
        for dst in dest_paths:
            dst_resolved = dst.resolve()
            if dst_resolved.exists() and dst_resolved not in source_set:
                blocking.append(str(dst_resolved))

        if blocking:
            raise RuntimeError(
                'Destination already exists and is not part of this rename set. Aborting. Example: '
                + ', '.join(blocking[:10])
            )

        staged = []
        completed = []
        try:
            for src, dst in zip(source_paths, dest_paths):
                tmp = src.with_name(f'.__rename_tmp__{uuid4().hex}{src.suffix}')
                src.rename(tmp)
                staged.append((tmp, src, dst))

            for tmp, src, dst in staged:
                tmp.rename(dst)
                completed.append((src, dst))

        except Exception as exc:
            print(f'Rename failed: {type(exc).__name__}: {exc}')
            print('Attempting rollback...')
            for src, dst in reversed(completed):
                if dst.exists() and not src.exists():
                    dst.rename(src)
            for tmp, src, dst in reversed(staged):
                if tmp.exists() and not src.exists():
                    tmp.rename(src)
            raise

        print(f'Rename complete. Files renamed: {len(completed)}')
        result_df = pd.DataFrame({
            'original_path': [str(src) for src, _ in completed],
            'new_path': [str(dst) for _, dst in completed],
            'original_name': [src.name for src, _ in completed],
            'new_name': [dst.name for _, dst in completed],
        })
        display(result_df.head(DISPLAY_LIMIT))


RuntimeError: Cannot rename while preview has errors. error_rows=11100